In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib
import time

print("Alle Pakete geladen ✓")

Alle Pakete geladen ✓


In [2]:
np.random.seed(42)
n = 1000

druck = np.random.normal(150, 20, n)
temperatur = np.random.normal(75, 10, n)
durchfluss = np.random.normal(50, 8, n)
vibration = np.random.normal(0.5, 0.1, n)
betriebsstunden = np.random.uniform(0, 10000, n)

ausfall = (
    0.3 * (temperatur / 100) +
    0.4 * (vibration / 1.0) +
    0.2 * (betriebsstunden / 10000) +
    0.1 * (druck / 200) +
    np.random.normal(0, 0.05, n)
).clip(0, 1)

df = pd.DataFrame({
    'Druck': druck,
    'Temperatur': temperatur,
    'Durchfluss': durchfluss,
    'Vibration': vibration,
    'Betriebsstunden': betriebsstunden,
    'Ausfall': ausfall
})

X = df.drop('Ausfall', axis=1)
y = df['Ausfall']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Daten geladen: {df.shape}")

Daten geladen: (1000, 6)


In [3]:
# LightGBM trainieren
start = time.time()

model_lgb = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
model_lgb.fit(X_train, y_train)

zeit_lgb = time.time() - start

# Ergebnisse
y_pred_lgb = model_lgb.predict(X_test)
rmse_lgb = mean_squared_error(y_test, y_pred_lgb) ** 0.5
r2_lgb = r2_score(y_test, y_pred_lgb)

print(f"LightGBM — RMSE: {rmse_lgb:.4f} | R²: {r2_lgb:.4f} | Zeit: {zeit_lgb:.2f}s")

LightGBM — RMSE: 0.0534 | R²: 0.6504 | Zeit: 3.91s


In [4]:
# XGBoost trainieren — zum Vergleich
start = time.time()

model_xgb = xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
model_xgb.fit(X_train, y_train)

zeit_xgb = time.time() - start

# Ergebnisse
y_pred_xgb = model_xgb.predict(X_test)
rmse_xgb = mean_squared_error(y_test, y_pred_xgb) ** 0.5
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost  — RMSE: {rmse_xgb:.4f} | R²: {r2_xgb:.4f} | Zeit: {zeit_xgb:.2f}s")
print(f"LightGBM — RMSE: {rmse_lgb:.4f} | R²: {r2_lgb:.4f} | Zeit: {zeit_lgb:.2f}s")

XGBoost  — RMSE: 0.0581 | R²: 0.5874 | Zeit: 0.15s
LightGBM — RMSE: 0.0534 | R²: 0.6504 | Zeit: 3.91s


In [5]:
# Benchmark Tabelle
ergebnisse = pd.DataFrame({
    'Modell': ['XGBoost v1', 'LightGBM'],
    'RMSE': [rmse_xgb, rmse_lgb],
    'R²': [r2_xgb, r2_lgb],
    'Zeit (s)': [zeit_xgb, zeit_lgb]
})

print(ergebnisse.to_string(index=False))

    Modell     RMSE       R²  Zeit (s)
XGBoost v1 0.058064 0.587366  0.153040
  LightGBM 0.053448 0.650373  3.913895
